#### This code authenticates and initializes the Google Earth Engine API and installs the earthengine-api and geemap libraries for visualizing geographical data in Jupyter Notebooks.

In [ ]:
!pip install geemap

In [ ]:
!pip install earthengine-api

In [2]:
import geemap
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project='my-project')

#### Part 1: 

In [ ]:
Map = geemap.Map()

# Define the bounding box for the area of interest
data = ee.Geometry.BBox(-112.5439, 34.0891, -85.0342, 49.6858)

# Center the map on the defined area
Map.centerObject(data)

# Load the Landsat 7 image
image = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")

# Visualization parameters for Landsat
landsat_vis = {"bands": ["B4", "B3", "B2"], "gamma": 1.4}

# Add the Landsat image to the map
Map.addLayer(image, landsat_vis, "LE7_TOA_5YEAR/1999_2003", True)

# Create the fishnet grid
fishnet = geemap.fishnet(data, h_interval=4.0, v_interval=4.0, delta=1)

# Style for the fishnet grid
grid_style = {'color': '#FFFF00', 'width': 2, 'fillColor': '#80808000'}

# Add the fishnet grid to the map
Map.addLayer(fishnet.style(**grid_style), {}, "Fishnet Grid")

# Display the map
Map


In [ ]:

import os

# Define the output directory for downloading the tiles
out_dir = os.path.expanduser("~/Downloads")

# Download image tiles using the fishnet grid
geemap.download_ee_image_tiles(
    image, fishnet, out_dir, prefix="landsat_tiles", crs="EPSG:3857", scale=30
)


#### Part 2: 

In [ ]:

# Load the US counties feature collection
counties = ee.FeatureCollection("TIGER/2018/Counties")

# Filter for Claiborne County (where Tazewell is located) in Tennessee
Claiborne_county = counties.filter(ee.Filter.And(
    ee.Filter.eq("STATEFP", "47"),  # Tennessee state code
    ee.Filter.eq("COUNTYFP", "025")  # Claiborne County (Tazewell is here)
))

# Convert the FeatureCollection to a GeoDataFrame
gdf = geemap.ee_to_gdf(Claiborne_county)

# Convert the GeoDataFrame back to an Earth Engine FeatureCollection
fc = geemap.gdf_to_ee(gdf)

# Check if the filtered collection is empty
if fc.size().getInfo() == 0:
    raise ValueError("The filtered collection is empty. Please check the filtering criteria.")

# Create a map
Map = geemap.Map()

# Add the county to the map
Map.addLayer(fc, {'color': 'red'}, 'Claiborne County (Tazewell)')

# Center the map on the county
Map.centerObject(fc, 10)

# Load the Landsat 8 image collection
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

# List of years
years = list(range(2017, 2024))

# Visualization parameters for Landsat
landsat_vis = {"bands": ["SR_B5", "SR_B6", "SR_B2"], "min": 0, "max": 30000, "gamma": 1}

# Function to mask clouds
def mask_clouds(image):
    # Cloud mask for Landsat 8
    qa = image.select('QA_PIXEL')
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)  # 3rd bit for clouds
    return image.updateMask(cloud_mask)

# Create and add annual cloud-free images to the map
for year in years:
    # Filter by year and county
    filtered = landsat.filterBounds(fc) \
                     .filterDate(f'{year}-01-01', f'{year}-12-31') \
                     .map(mask_clouds) \
                     .median()  # Use median to reduce clouds

    # Clip the image to the county boundary
    clipped_image = filtered.clip(fc)

    # Add the clipped image to the map with the visualization parameters
    Map.addLayer(clipped_image, landsat_vis, f'Landsat {year}')

# Add a text label to the map
Map.add_text("Made by Clara MR", fontsize=20, position='bottomright')

# Display the map
Map


In [ ]:
import os

# Define the download folder
out_dir = os.path.expanduser("~/Downloads")  # Download to the user's "Downloads" folder

# Download annual clipped images
for year in years:
    # Filter by year and county
    filtered = landsat.filterBounds(fc) \
                     .filterDate(f'{year}-01-01', f'{year}-12-31') \
                     .map(mask_clouds) \
                     .median()  # Use median to reduce clouds

    # Clip the image to the county boundary
    clipped_image = filtered.clip(fc)

    # Define the full file path
    filename = os.path.join(out_dir, f'landsat_{year}_tazewell.tif')

    # Download the clipped image
    geemap.download_ee_image(
        clipped_image,
        filename=filename,  # Full file path
        scale=30,           # 30-meter resolution
        region=fc.geometry().bounds(),  # Region of interest (county boundaries)
        crs="EPSG:4326",    # Coordinate reference system (WGS84)
        overwrite=True      # Overwrite existing files if they already exist
    )


#### Part 3: 

In [ ]:
# Load the correct county (Claiborne County, where Tazewell is located)
counties = ee.FeatureCollection("TIGER/2018/Counties")
Tazewell_county = counties.filter(ee.Filter.And(
    ee.Filter.eq("STATEFP", "47"),  # Tennessee
    ee.Filter.eq("COUNTYFP", "025")  # Claiborne County
))

# Create a map
Map = geemap.Map()
Map.centerObject(Tazewell_county, 10)
Map.addLayer(Tazewell_county, {'color': 'red'}, 'Claiborne County (Tazewell)')

# Load Sentinel-2 images
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

# Function to mask clouds
def mask_clouds(image):
    cloud_prob = image.select('MSK_CLDPRB')
    cloud_mask = cloud_prob.lt(20)  # Cloud mask for probabilities > 20%
    return image.updateMask(cloud_mask)

# Function to calculate NDVI
def calculate_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')  # NDVI calculation
    return image.addBands(ndvi)

# Define the years to analyze
years = list(range(2018, 2024))

# Process each year
for year in years:
    # Filter by year and summer months (June to August)
    filtered = sentinel2.filterBounds(Tazewell_county) \
                        .filterDate(f'{year}-06-01', f'{year}-08-31') \
                        .map(mask_clouds) \
                        .map(calculate_ndvi)

    # Check if there are any images
    if filtered.size().getInfo() == 0:
        print(f"No images for {year}.")
        continue

    # Calculate the median and clip
    median_image = filtered.median().clip(Tazewell_county)

    # Add NDVI to the map
    Map.addLayer(median_image.select('NDVI'), {'min': 0, 'max': 1, 'palette': ['red', 'yellow', 'green']}, f'NDVI {year}')

# Add a text label to the map
Map.add_text("By Clara MR", fontsize=20, position='bottomright')

# Display the map
Map



In [ ]:
import os

# Define the download folder
out_dir = os.path.expanduser("~/Downloads")  # Download to the user's "Downloads" folder

# Download annual clipped images
for year in years:
    # Filter by year and county
    filtered = sentinel2.filterBounds(Tazewell_county) \
                        .filterDate(f'{year}-01-01', f'{year}-12-31') \
                        .map(mask_clouds) \
                        .median()  # Use median to reduce clouds

    # Clip the image to the county boundary
    clipped_image = filtered.clip(Tazewell_county)

    # Define the full file path
    filename = os.path.join(out_dir, f'sentinel2_{year}_tazewell.tif')

    # Download the clipped image
    geemap.download_ee_image(
        clipped_image,
        filename=filename,  # Full file path
        scale=30,           # 30-meter resolution
        region=Tazewell_county.geometry().bounds(),  # Region of interest (county boundaries)
        crs="EPSG:4326",    # Coordinate reference system (WGS84)
        overwrite=True      # Overwrite existing files if they already exist
    )

print("Download completed.")


#### Part 4:

In [ ]:
# Load the US counties feature collection
counties = ee.FeatureCollection("TIGER/2018/Counties")

# Filter for Claiborne County (where Tazewell is located)
Tazewell_county = counties.filter(ee.Filter.And(
    ee.Filter.eq("STATEFP", "47"),  # Tennessee
    ee.Filter.eq("COUNTYFP", "025")  # Claiborne County
))

# Convert to a FeatureCollection for Earth Engine
gdf = geemap.ee_to_gdf(Tazewell_county)
fc = geemap.gdf_to_ee(gdf)

# Check if the filtered collection is empty
if fc.size().getInfo() == 0:
    raise ValueError("Error: The filtered county is empty. Check the FIPS codes.")

# Create an interactive map
Map = geemap.Map()
Map.centerObject(fc, 10)
Map.addLayer(fc, {'color': 'blue'}, 'Tazewell County')

# Load the NAIP image collection
naip = ee.ImageCollection("USDA/NAIP/DOQQ")

# List of years (2010-2023)
years = list(range(2010, 2024))

# Visualization parameters for **False Color (NIR-R-G)**
naip_vis = {"bands": ["N", "R", "G"], "min": 0, "max": 255, "gamma": 1.4}

# Process annual NAIP images
for year in years:
    filtered = naip.filterBounds(fc) \
                   .filterDate(f'{year}-01-01', f'{year}-12-31') \
                   .median()  # Use median to reduce clouds

    clipped_image = filtered.clip(fc)

    # Add to the map with **NIR-R-G** combination
    Map.addLayer(clipped_image, naip_vis, f'NAIP {year} (False Color)')

# Add text to the map
Map.add_text("Made by Clara MR", fontsize=20, position='bottomright')

# Display the interactive map
Map



In [ ]:
import os

# Define output folder
out_dir = os.path.expanduser("~/Downloads/NAIP_Images")

# Create the output folder if it doesn't exist
os.makedirs(out_dir, exist_ok=True)

# Download images by year
for year in years:
    naip_image = get_naip(year)  # Function to retrieve NAIP image for the year

    # Define file name
    filename = os.path.join(out_dir, f'NAIP_{year}_county.tif')

    # Download the image with reduced resolution for faster download
    geemap.download_ee_image(
        naip_image, 
        filename=filename,
        scale=10,  # 10m resolution for faster download
        region=fc.geometry().bounds(),  # Region to download (county boundaries)
        crs="EPSG:4326",  # Coordinate reference system (WGS84)
        overwrite=True  # Overwrite files if they already exist
    )

    print(f"Image {year} downloaded to: {filename}")



#### Part 5:

In [ ]:
import os

# Load the US counties feature collection
counties = ee.FeatureCollection("TIGER/2018/Counties")

# Filter for Claiborne County in Tennessee (STATEFP = 47, COUNTYFP = 025)
county_name = "Claiborne"
Tennessee_county = counties.filter(ee.Filter.eq("NAME", county_name))

# Convert to GeoDataFrame
gdf = geemap.ee_to_gdf(Tennessee_county)

# Define output folder
out_dir = os.path.expanduser("~/Downloads")
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# Save as Shapefile
shapefile_path = os.path.join(out_dir, "Claiborne_County.shp")
gdf.to_file(shapefile_path)

# Save as GeoJSON
geojson_path = os.path.join(out_dir, "Claiborne_County.geojson")
gdf.to_file(geojson_path, driver="GeoJSON")

print(f"Shapefile saved at: {shapefile_path}")
print(f"GeoJSON saved at: {geojson_path}")
